# Summary
Script to run against **one** partner file (CNAF, MSA or CNOUS) that has already gone through
its own cleaning/formatting notebook.

This notebook:
- loads a single partner export, no merging between partners nor with the previous year,
- sets the default column values: exercice_id, uuid_doc, zrr, qpv, a_valider, refuser, created_at, updated_at,
- generates unique codes and assigns them to the "id_psp" column,
- writes the enriched rows to a **new** CSV, leaving the input file untouched,
- appends the codes it just used to `EXISTING_CODES_PATHFILE_2026`.

`EXISTING_CODES_PATHFILE_2026` is a single column ("code") CSV holding every code already handed
out for this exercice. It is read before generating anything and rewritten at the end, which is
what keeps codes unique across the successive runs of this notebook (one per partner file, one
per CNOUS wave, ...).

Pick the file to process with the `SOURCE` variable in the configuration cell below.

In [ ]:
import os
import csv
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv

# utils lives at the data/ root: make that root importable first, since this notebook
# runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

from utils.data_utils import get_current_date_for_file_name

load_dotenv()

In [ ]:
# Which cleaned partner file to process: 'CNAF', 'MSA' or 'CNOUS'.
SOURCE = 'CNAF'

SOURCE_INPUT_ENV_VAR = {
    'CNAF': 'DB_CNAF_EXPORT_2026',
    'MSA': 'DB_MSA_EXPORT_2026',
    'CNOUS': 'DB_CNOUS_EXPORT_2026',
}

input_filepath = os.environ[SOURCE_INPUT_ENV_VAR[SOURCE]]
existing_codes_filepath = os.environ['EXISTING_CODES_PATHFILE_2026']

# The input file is never modified in place: rows enriched with their default columns and
# their id_psp go to a dated file sitting next to it.
output_filepath = str(
    Path(input_filepath).with_name(
        get_current_date_for_file_name(f"{SOURCE.lower()}-with-codes.csv")
    )
)

print(f"input:          {input_filepath}")
print(f"existing codes: {existing_codes_filepath}")
print(f"output:         {output_filepath}")

In [ ]:
# keep_default_na is necessary otherwise string such as "NA" is considered as NaN...
df = pd.read_csv(
    input_filepath, sep=';', encoding='utf-8', dtype=str, keep_default_na=False,
    quoting=csv.QUOTE_ALL,
)

len(df)

In [ ]:
# Always try to load the codes already used this year, so the ones generated below can
# never collide with them. The file may not exist yet on the very first run.
if Path(existing_codes_filepath).exists():
    df_existing_codes = pd.read_csv(
        existing_codes_filepath, sep=',', encoding='utf-8', dtype=str,
        keep_default_na=False, engine='c', on_bad_lines='skip',
    )
else:
    print(f"{existing_codes_filepath} does not exist yet, starting from an empty code list")
    df_existing_codes = pd.DataFrame({'code': pd.Series(dtype=str)})

existing_codes = set(df_existing_codes['code'])

len(existing_codes)

In [ ]:
assert(len(df[df['nom'].isna() | df['prenom'].isna()]) == 0)
assert('id_psp' not in df.columns or df['id_psp'].eq('').all())

In [ ]:
# Add missing default column needed to production data
exercice_2026 = 5

timestamp_with_custom_tz = pd.Timestamp.now(tz='Europe/Paris')

df['exercice_id'] = exercice_2026
df['uuid_doc'] = np.NaN
df[['zrr', 'qpv', 'a_valider', 'refuser']] = False
df[['created_at', 'updated_at']] = timestamp_with_custom_tz

In [ ]:
# Unique codes generation
import random
import string
import datetime

current_date = datetime.datetime.now()
current_year = str(current_date.year)[-2:]

def get_characters_set(size = 4):
    return ''.join(random.choices([c for c in string.ascii_uppercase if c not in 'OI'], k=size))

def generate_code():
    return f"{current_year}-{get_characters_set(4)}-{get_characters_set(4)}"

# init set of codes with the existing ones: a generated code that is already taken is
# swallowed by the set, so the loop keeps going until we have one new code per row
unique_codes = set(existing_codes)

while len(unique_codes) < (len(df) + len(existing_codes)):
    unique_codes.add(generate_code())

new_codes = unique_codes.difference(existing_codes)

# Ensure we have generated codes for all the rows
assert len(new_codes) == len(df)

In [ ]:
# Assign generated code for production data
df['id_psp'] = list(new_codes)

In [ ]:
print(f"{len(new_codes)} unique codes generated for {SOURCE}")
print(f"{len(df)} benefs from {SOURCE}")
print(f"{len(df[df['genre'] == 'M'])} M benefs from {SOURCE}")
print(f"{len(df[df['genre'] == 'F'])} F benefs from {SOURCE}")

In [ ]:
df[['organisme', 'situation']].value_counts()

In [ ]:
df.to_csv(output_filepath, sep=';', index=False, encoding='utf-8')

In [ ]:
# Keep track of the codes just handed out, so the next run of this notebook (other partner
# file, next CNOUS wave, ...) keeps generating unique ones.
# Run this cell once, and only after the output file above has been written.
df_updated_codes = pd.DataFrame({'code': sorted(unique_codes)})

assert len(df_updated_codes) == len(existing_codes) + len(df)

df_updated_codes.to_csv(existing_codes_filepath, sep=',', index=False, encoding='utf-8')

print(f"{len(df_updated_codes)} codes now tracked in {existing_codes_filepath}")